**Creación del CRM de retención**

In [1]:
import pandas as pd
import sqlite3

print("--- FASE 1: CREACIÓN DEL CRM DE RETENCIÓN ---")

# 1. Cargamos el archivo CSV con los resultados de riesgo calculados por el modelo predictivo
df_riesgo = pd.read_csv("../../datos/procesados/socios_prioritarios_ia.csv")

# 2. Ordenamos de mayor a menor riesgo y nos quedamos unicamente con los 50 casos mas urgentes
df_top_50 = df_riesgo.sort_values(by='probabilidad_churn', ascending=False).head(50)

ruta_db = '../../datos/crm_gimnasio.db'
conexion = sqlite3.connect(ruta_db)

# 3. Guardamos estos 50 socios en una tabla nueva dentro de nuestra base de datos local
df_top_50.to_sql('campana_retencion', conexion, if_exists='replace', index=False)

print(f"Base de datos SQLite creada en: {ruta_db}")
print(f"Se han cargado los {len(df_top_50)} socios más críticos en la tabla 'campana_retencion'.")

# 4. Hacemos una consulta rapida de lectura para verificar que la tabla se ha creado bien
df_verificacion = pd.read_sql("SELECT * FROM campana_retencion LIMIT 5", conexion)
print("\nMuestra de los 5 socios con más riesgo:")
display(df_verificacion)

conexion.close()

--- FASE 1: CREACIÓN DEL CRM DE RETENCIÓN ---
Base de datos SQLite creada en: ../../datos/crm_gimnasio.db
Se han cargado los 50 socios más críticos en la tabla 'campana_retencion'.

Muestra de los 5 socios con más riesgo:


,id_socio,probabilidad_churn,zona_proximidad,total_visitas,dias_sin_venir,edad,tipo_socio
0,4374.0,0.996239,1,191.0,186.0,40.0,AB. FAMILIAR
1,25538.0,0.991919,0,5.0,689.0,31.0,AB. FAMILIAR
2,23983.0,0.991425,2,83.0,216.0,17.0,AB. JOVE 14-17
3,27607.0,0.990425,0,51.0,322.0,25.0,AB. ATURAT
4,21984.0,0.988657,0,46.0,511.0,20.0,AB. FAMILIAR


**Prueba de Generación de Email Aleatorio**

In [2]:
import sqlite3
import pandas as pd
from openai import OpenAI

# 1. Conexión al servidor local de LM Studio
client = OpenAI(base_url="http://localhost:11434/v1", api_key="lm-studio")

# 2. Selección de un socio aleatorio
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
socio_aleatorio = pd.read_sql("SELECT * FROM campana_retencion ORDER BY RANDOM() LIMIT 1", conexion).iloc[0]
conexion.close()

id_socio = int(socio_aleatorio['id_socio'])
dias_sin_venir = int(socio_aleatorio['dias_sin_venir'])
zona = int(socio_aleatorio['zona_proximidad'])
visitas = int(socio_aleatorio['total_visitas'])
edad = int(socio_aleatorio['edad'])
cuota = str(socio_aleatorio['tipo_socio'])

# 3. Reglas de negocio y preprocesamiento lógico
# Lógica de distancia
if zona == 0:
    texto_ubicacion = "Vive muy CERCA del gimnasio."
    regla_distancia = "Destaca la comodidad de tener el centro a un paso de casa."
else:
    texto_ubicacion = "Vive LEJOS del gimnasio."
    regla_distancia = "Empatiza con el desplazamiento y ofrece opciones virtuales."

# Lógica de franjas para las visitas
if visitas < 10:
    regla_visitas = "Anímale a retomar el impulso inicial. Recuérdale que los comienzos cuestan, pero lo importante es volver a dar el primer paso."
elif visitas <= 50:
    regla_visitas = "Menciónale que tenía un ritmo estupendo y anímale a recuperar esa buena rutina de entrenamiento que ya había conseguido."
else:
    regla_visitas = "Hazle saber que valoramos muchísimo su larga y constante trayectoria con nosotros. Anímale a no perder todo ese progreso acumulado."

# 4. Prompt Maestro
prompt = f"""
Actúa como el Director de Fidelización del centro deportivo Can Xaubet.
Escribe un email persuasivo y empático para recuperar al socio con ID {id_socio}.

CONTEXTO DEL SOCIO (INFORMACIÓN INTERNA):
- Días de ausencia: {dias_sin_venir} días.
- Ubicación: {texto_ubicacion}
- Edad: {edad} años.

REGLAS DE REDACCIÓN (IMPORTANTE):
1. SALUDO Y PRIVACIDAD: Usa "Estimado/a socio/a," (prohibido usar el ID). NUNCA menciones la cantidad exacta de días de ausencia.
2. TONO ADAPTADO A LA EDAD: Ajusta la madurez y energía del mensaje sabiendo que tiene {edad} años. 
3. APELA A SU HISTORIAL: {regla_visitas} ESTÁ TERMINANTEMENTE PROHIBIDO mencionar el número exacto de visitas que ha hecho.
4. PERSONALIZACIÓN POR DISTANCIA: {regla_distancia}
5. FLEXIBILIDAD DE CUOTA: Recuérdale que si sus circunstancias han cambiado, estamos a su disposición para adaptar su cuota o tarifa a sus necesidades actuales. NUNCA uses nombres técnicos de cuotas.
6. REGALO: Ofrécele una sesión gratuita de 30 min con un entrenador personal para reevaluar objetivos.
7. CIERRE: Despídete como "Director de Fidelización, Can Xaubet".
8. LÍMITE: Máximo 150 palabras.
"""

print(f"Extrayendo socio al azar de la base de datos...")
print(f"Contexto de la IA: {dias_sin_venir} días sin venir, Zona {zona}, Edad {edad}, Visitas {visitas}, Cuota {cuota}\n")

# 5. Generación del correo (Llamada al modelo local)
try:
    respuesta = client.chat.completions.create(
        model="llama3.1",
        messages=[
            {"role": "system", "content": "Eres un experto en redacción corporativa y retención de clientes."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    print("EMAIL GENERADO ALEATORIAMENTE (VÍA LLAMA 3.1):")
    print("-" * 50)
    print(respuesta.choices[0].message.content)
    print("-" * 50)
    
except Exception as e:
    print(f"Error de conexión: {e}")
    print("Por favor, verifica que LM Studio está abierto y el servidor está activado.")

Extrayendo socio al azar de la base de datos...
Contexto de la IA: 129 días sin venir, Zona 0, Edad 21, Visitas 95, Cuota FAMILIA NOMBROSA

EMAIL GENERADO ALEATORIAMENTE (VÍA LLAMA 3.1):
--------------------------------------------------
Asunto: ¡De vuelta a la acción!

Estimado socio,

Esperamos que estés bien y disfrutando del verano. Queremos hacerte llegar un mensaje de apoyo y recordarte lo importante que eres para nuestra comunidad deportiva. Después de una temporada con poca frecuencia en nuestros instalaciones, estamos emocionados por la oportunidad de verte de nuevo.

Sabemos que hay muchos factores que pueden influir en nuestras agendas, pero no queríamos dejar pasar la ocasión de recordarte lo cerca que estás de casa y cómo esto te permite disfrutar al máximo de tus sesiones con nosotros. Valoramos enormemente tu dedicación y progreso durante años y no queremos verlo perderse.

Si debido a cambios en tu vida necesitas ajustar tu compromiso, estamos aquí para adaptarnos a ti.

**Generación masiva de correos para los 50 usuarios con más probabilidad de darse de baja**

In [ ]:
import sqlite3
import pandas as pd
from openai import OpenAI
import time

# 1. Conexión al servidor local de Ollama
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 2. Carga de datos del CRM
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
df_campana = pd.read_sql("SELECT * FROM campana_retencion", conexion)

lista_emails = []

print(f"--- INICIANDO GENERACIÓN LOCAL MASIVA ---")
print(f"Procesando {len(df_campana)} socios con Llama 3.1...")
print("Este proceso puede tardar varios minutos...\n")

contador = 1 

# 3. Bucle de generación masiva
for index, socio in df_campana.iterrows():
    id_socio = int(socio['id_socio'])
    dias_sin_venir = int(socio['dias_sin_venir'])
    zona = int(socio['zona_proximidad'])
    visitas = int(socio['total_visitas'])
    edad = int(socio['edad'])
        
    # Lógica de distancia
    if zona == 0:
        texto_ubicacion = "Vive muy CERCA del gimnasio."
        regla_distancia = "Destaca la comodidad de tener el centro a un paso de casa."
    else:
        texto_ubicacion = "Vive LEJOS del gimnasio."
        regla_distancia = "Empatiza con el desplazamiento y ofrece opciones virtuales."

    # Lógica de franjas para las visitas
    if visitas < 10:
        regla_visitas = "Anímale a retomar el impulso inicial. Recuérdale que los comienzos cuestan, pero lo importante es volver a dar el primer paso."
    elif visitas <= 50:
        regla_visitas = "Menciónale que tenía un ritmo estupendo y anímale a recuperar esa buena rutina de entrenamiento que ya había conseguido."
    else:
        regla_visitas = "Hazle saber que valoramos muchísimo su larga y constante trayectoria con nosotros. Anímale a no perder todo ese progreso acumulado."

    # --- EL PROMPT MAESTRO ---
    prompt = f"""
    Actúa como el Director de Fidelización del centro deportivo Can Xaubet.
    Escribe un email persuasivo y empático para recuperar al socio con ID {id_socio}.

    CONTEXTO DEL SOCIO (INFORMACIÓN INTERNA):
    - Días de ausencia: {dias_sin_venir} días.
    - Ubicación: {texto_ubicacion}
    - Edad: {edad} años.

    REGLAS DE REDACCIÓN (IMPORTANTE):
    1. SALUDO Y PRIVACIDAD: Usa "Estimado/a socio/a," (prohibido usar el ID). NUNCA menciones la cantidad exacta de días de ausencia.
    2. TONO ADAPTADO A LA EDAD: Ajusta la madurez y energía del mensaje sabiendo que tiene {edad} años. 
    3. APELA A SU HISTORIAL: {regla_visitas} ESTÁ TERMINANTEMENTE PROHIBIDO mencionar el número exacto de visitas que ha hecho.
    4. PERSONALIZACIÓN POR DISTANCIA: {regla_distancia}
    5. FLEXIBILIDAD DE CUOTA: Recuérdale que si sus circunstancias han cambiado, estamos a su disposición para adaptar su cuota o tarifa a sus necesidades actuales. NUNCA uses nombres técnicos de cuotas.
    6. REGALO: Ofrécele una sesión gratuita de 30 min con un entrenador personal para reevaluar objetivos.
    7. CIERRE: Despídete como "Director de Fidelización, Can Xaubet".
    8. LÍMITE: Máximo 150 palabras.
    """

    # --- LLAMADA AL MODELO ---
    try:
        respuesta = client.chat.completions.create(
            model="llama3.1", 
            messages=[
                {"role": "system", "content": "Eres un experto en redacción corporativa y retención de clientes."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        email_texto = respuesta.choices[0].message.content
        
        print(f"[{contador}] [OK] -> Correo generado para el Socio {id_socio}")
        
    except Exception as e:
        email_texto = f"Error en generación: {e}"
        print(f"[{contador}] [ERROR] -> Fallo en el Socio {id_socio}")

    lista_emails.append(email_texto)
    
    contador += 1 

# 4. Actualización de la Base de Datos
df_campana['email_propuesto'] = lista_emails
df_campana.to_sql('campana_retencion', conexion, if_exists='replace', index=False)
conexion.close()

print("\n--- ¡PROCESO FINALIZADO CON ÉXITO! ---")
print("Base de datos crm_gimnasio.db actualizada con los 50 correos de IA Local.")

--- INICIANDO GENERACIÓN LOCAL MASIVA ---
Procesando 50 socios con Llama 3.1...
Este proceso puede tardar varios minutos...



**Comprobación Final de la Base de Datos**

In [5]:
import sqlite3
import pandas as pd

# 1. Establecemos la conexion directa con nuestra base de datos local SQLite
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')

# 2. Ejecutamos una consulta SQL seleccionando unicamente las columnas clave
# 3. Limitamos la extraccion a 5 socios para hacer una validacion rapida sin saturar la pantalla
df_resultado = pd.read_sql("SELECT id_socio, edad, total_visitas, zona_proximidad, email_propuesto FROM campana_retencion LIMIT 5", conexion)

conexion.close()

# 4. Renderizamos la tabla resultante para verificar visualmente la integracion de los textos
display(df_resultado)

,id_socio,edad,total_visitas,zona_proximidad,email_propuesto
0,4374.0,40.0,191.0,1,Asunto: Un saludo y un recordatorio con cariño...
1,25538.0,31.0,5.0,0,"Estimado socio,\n\nEspero que estés bien. Me a..."
2,23983.0,17.0,83.0,2,"Estimado socio,\n\nMe alegra ver tu nombre en ..."
3,27607.0,25.0,51.0,0,Asunto: ¡No te pierdas todo lo que has trabaja...
4,21984.0,20.0,46.0,0,Asunto: ¡Volvamos a entrenar juntos!\n\nEstima...


**Exportación final para Power BI**

In [6]:
import sqlite3
import pandas as pd

# 1. Conectamos a la base de datos donde la IA guardó los correos
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')

# 2. Leemos la tabla completa de la campaña
df_final_powerbi = pd.read_sql("SELECT * FROM campana_retencion", conexion)

# 3. La exportamos a la carpeta de procesados para Power BI
ruta_pbi = '../../datos/procesados/campana_retencion_final.csv'
df_final_powerbi.to_csv(ruta_pbi, index=False)

conexion.close()

print(f"El archivo para Power BI se ha guardado en:\n{ruta_pbi}")

El archivo para Power BI se ha guardado en:
../../datos/procesados/campana_retencion_final.csv
